In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import kagglehub
import os
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [3]:
# download the data from kaggle
import kagglehub

# Download latest version
dataset_path = kagglehub.dataset_download("philiphyde1/nfl-stats-1999-2022")

100%|██████████| 48.2M/48.2M [00:00<00:00, 180MB/s]

Extracting files...


In [4]:
print("Files in the dataset folder:", os.listdir(dataset_path))

Files in the dataset folder: ['yearly_team_stats_offense.csv', 'yearly_team_stats_defense.csv', 'weekly_player_stats_offense.csv', 'weekly_team_stats_offense.csv', 'yearly_player_stats_defense.csv', 'weekly_team_stats_defense.csv', 'weekly_player_stats_defense.csv', 'yearly_player_stats_offense.csv']


In [5]:
# Load the CSV file into a DataFrame
# [weekly_player_stats_defense.csv'', 'weekly_player_stats_defense.csv', 'weekly_team_stats_defense.csv', 'weekly_team_stats_defense.csv', 'yearly_player_stats_defense.csv', 'yearly_player_stats_defense.csv', 'yearly_team_stats_defense.csv', 'yearly_team_stats_defense.csv']
weekly_df_defense = pd.read_csv(os.path.join(dataset_path, 'weekly_team_stats_defense.csv'))

weekly_df_offense = pd.read_csv(os.path.join(dataset_path, 'weekly_team_stats_offense.csv'))

In [7]:
weekly_df_defense.head()

,game_id,season,week,team,season_type,safety,interception,fumble,fumble_lost,fumble_forced,...,average_solo_tackle,average_assist_tackle,average_tackle_with_assist,average_sack,average_qb_hit,average_def_touchdown,average_defensive_two_point_attempt,average_defensive_two_point_conv,average_defensive_extra_point_attempt,average_defensive_extra_point_conv
0,2012_01_SEA_ARI,2012,1,ARI,REG,0,1,2,1,1,...,55.000000,6.0,4.000000,3.0,8.000000,0.000000,0.0,0.0,0,0
1,2012_02_ARI_NE,2012,2,ARI,REG,0,1,0,0,0,...,51.000000,7.0,2.000000,3.5,7.000000,0.000000,0.0,0.0,0,0
2,2012_03_PHI_ARI,2012,3,ARI,REG,0,0,2,2,2,...,49.333333,6.0,1.666667,4.0,9.333333,0.333333,0.0,0.0,0,0
3,2012_04_MIA_ARI,2012,4,ARI,REG,0,2,4,2,3,...,50.500000,6.5,2.500000,4.0,9.500000,0.250000,0.0,0.0,0,0
4,2012_05_ARI_STL,2012,5,ARI,REG,0,1,0,0,0,...,47.400000,6.2,2.000000,3.4,8.800000,0.200000,0.0,0.0,0,0


In [8]:
num_columns_df = weekly_df_defense.shape[1]
num_rows_df = weekly_df_defense.shape[0]
column_names_df = weekly_df_defense.columns.tolist()

print(f"The dataset has {num_columns_df} columns and {num_rows_df} rows")

for col in column_names_df:
    null_count = weekly_df_defense[col].isnull().sum()  # Count null values
    print(f"Column: {col}, "
          f"Type: {weekly_df_defense[col].dtype}, "
          f"Unique Values: {weekly_df_defense[col].nunique()}, "
          f"Null Values: {null_count} ({null_count/num_rows_df:.1%})")

The dataset has 65 columns and 7088 rows
Column: game_id, Type: object, Unique Values: 3544, Null Values: 0 (0.0%)
Column: season, Type: int64, Unique Values: 13, Null Values: 0 (0.0%)
Column: week, Type: int64, Unique Values: 22, Null Values: 0 (0.0%)
Column: team, Type: object, Unique Values: 32, Null Values: 0 (0.0%)
Column: season_type, Type: object, Unique Values: 2, Null Values: 0 (0.0%)
Column: safety, Type: int64, Unique Values: 3, Null Values: 0 (0.0%)
Column: interception, Type: int64, Unique Values: 7, Null Values: 0 (0.0%)
Column: fumble, Type: int64, Unique Values: 8, Null Values: 0 (0.0%)
Column: fumble_lost, Type: int64, Unique Values: 6, Null Values: 0 (0.0%)
Column: fumble_forced, Type: int64, Unique Values: 7, Null Values: 0 (0.0%)
Column: fumble_not_forced, Type: int64, Unique Values: 6, Null Values: 0 (0.0%)
Column: fumble_out_of_bounds, Type: int64, Unique Values: 5, Null Values: 0 (0.0%)
Column: solo_tackle, Type: int64, Unique Values: 56, Null Values: 0 (0.0%)
Co

In [9]:
num_columns_of = weekly_df_offense.shape[1]
num_rows_of = weekly_df_offense.shape[0]
column_names_of = weekly_df_offense.columns.tolist()

print(f"The dataset has {num_columns_of} columns and {num_rows_of} rows")

for col in column_names_of:
    null_count = weekly_df_offense[col].isnull().sum()  # Count null values
    print(f"Column: {col}, "
          f"Type: {weekly_df_offense[col].dtype}, "
          f"Unique Values: {weekly_df_offense[col].nunique()}, "
          f"Null Values: {null_count} ({null_count/num_rows_of:.1%})")

The dataset has 115 columns and 7088 rows
Column: game_id, Type: object, Unique Values: 3544, Null Values: 0 (0.0%)
Column: season, Type: int64, Unique Values: 13, Null Values: 0 (0.0%)
Column: week, Type: int64, Unique Values: 22, Null Values: 0 (0.0%)
Column: team, Type: object, Unique Values: 32, Null Values: 0 (0.0%)
Column: season_type, Type: object, Unique Values: 2, Null Values: 0 (0.0%)
Column: shotgun, Type: int64, Unique Values: 87, Null Values: 0 (0.0%)
Column: no_huddle, Type: int64, Unique Values: 70, Null Values: 0 (0.0%)
Column: qb_dropback, Type: int64, Unique Values: 63, Null Values: 0 (0.0%)
Column: qb_scramble, Type: int64, Unique Values: 12, Null Values: 0 (0.0%)
Column: total_off_yards, Type: int64, Unique Values: 461, Null Values: 0 (0.0%)
Column: pass_attempts, Type: int64, Unique Values: 62, Null Values: 0 (0.0%)
Column: complete_pass, Type: int64, Unique Values: 45, Null Values: 0 (0.0%)
Column: incomplete_pass, Type: int64, Unique Values: 32, Null Values: 0 (0

In [ ]:
weekly_df_defense.head()

,game_id,season,week,team,season_type,safety,interception,fumble,fumble_lost,fumble_forced,...,average_solo_tackle,average_assist_tackle,average_tackle_with_assist,average_sack,average_qb_hit,average_def_touchdown,average_defensive_two_point_attempt,average_defensive_two_point_conv,average_defensive_extra_point_attempt,average_defensive_extra_point_conv
0,2012_01_SEA_ARI,2012,1,ARI,REG,0,1,2,1,1,...,55.000000,6.0,4.000000,3.0,8.000000,0.000000,0.0,0.0,0,0
1,2012_02_ARI_NE,2012,2,ARI,REG,0,1,0,0,0,...,51.000000,7.0,2.000000,3.5,7.000000,0.000000,0.0,0.0,0,0
2,2012_03_PHI_ARI,2012,3,ARI,REG,0,0,2,2,2,...,49.333333,6.0,1.666667,4.0,9.333333,0.333333,0.0,0.0,0,0
3,2012_04_MIA_ARI,2012,4,ARI,REG,0,2,4,2,3,...,50.500000,6.5,2.500000,4.0,9.500000,0.250000,0.0,0.0,0,0
4,2012_05_ARI_STL,2012,5,ARI,REG,0,1,0,0,0,...,47.400000,6.2,2.000000,3.4,8.800000,0.200000,0.0,0.0,0,0


In [ ]:
weekly_df_offense.head()

,game_id,season,week,team,season_type,shotgun,no_huddle,qb_dropback,qb_scramble,total_off_yards,...,average_fourth_down_failed,average_rush_touchdown,average_pass_touchdown,average_safety,average_interception,average_fumble,average_fumble_lost,average_fumble_forced,average_fumble_not_forced,average_fumble_out_of_bounds
0,2012_01_SEA_ARI,2012,1,ARI,REG,31,0,38,1,258,...,0.000000,1.000000,1.000000,0.0,1.000000,2.000000,1.000000,2.000000,0.0,0.0
1,2012_02_ARI_NE,2012,2,ARI,REG,25,7,30,1,242,...,0.000000,1.000000,1.000000,0.0,0.500000,2.000000,1.500000,2.000000,0.0,0.0
2,2012_03_PHI_ARI,2012,3,ARI,REG,31,0,29,2,321,...,0.333333,0.666667,1.333333,0.0,0.333333,1.666667,1.333333,1.666667,0.0,0.0
3,2012_04_MIA_ARI,2012,4,ARI,REG,52,5,56,0,352,...,0.250000,0.500000,1.750000,0.0,0.750000,1.500000,1.000000,1.500000,0.0,0.0
4,2012_05_ARI_STL,2012,5,ARI,REG,56,7,59,1,334,...,0.600000,0.400000,1.400000,0.0,0.600000,1.400000,1.000000,1.400000,0.0,0.0


In [11]:
wdf_def_2024 = weekly_df_defense[
    (weekly_df_defense['season'].isin([2024])) ]  # Using .copy() to avoid SettingWithCopyWarning
wdf_def_2024.head()

,game_id,season,week,team,season_type,safety,interception,fumble,fumble_lost,fumble_forced,...,average_solo_tackle,average_assist_tackle,average_tackle_with_assist,average_sack,average_qb_hit,average_def_touchdown,average_defensive_two_point_attempt,average_defensive_two_point_conv,average_defensive_extra_point_attempt,average_defensive_extra_point_conv
6518,2024_01_ARI_BUF,2024,1,ARI,REG,0,0,1,1,1,...,28.000000,22.000000,14.000000,2.00,3.000000,0.0,0.0,0.0,0,0
6519,2024_02_LA_ARI,2024,2,ARI,REG,0,0,1,1,1,...,29.000000,18.500000,10.000000,3.50,6.000000,0.0,0.0,0.0,0,0
6520,2024_03_DET_ARI,2024,3,ARI,REG,0,1,0,0,0,...,30.333333,20.666667,9.333333,3.00,4.666667,0.0,0.0,0.0,0,0
6521,2024_04_WAS_ARI,2024,4,ARI,REG,0,1,0,0,0,...,33.500000,19.250000,8.250000,2.25,3.750000,0.0,0.0,0.0,0,0
6522,2024_05_ARI_SF,2024,5,ARI,REG,0,2,1,1,1,...,34.000000,18.000000,7.200000,2.20,4.000000,0.0,0.0,0.0,0,0


In [12]:
wdf_of_2024 = weekly_df_offense[
    (weekly_df_offense['season'].isin([2024])) ]
wdf_of_2024.head()

,game_id,season,week,team,season_type,shotgun,no_huddle,qb_dropback,qb_scramble,total_off_yards,...,average_fourth_down_failed,average_rush_touchdown,average_pass_touchdown,average_safety,average_interception,average_fumble,average_fumble_lost,average_fumble_forced,average_fumble_not_forced,average_fumble_out_of_bounds
6518,2024_01_ARI_BUF,2024,1,ARI,REG,51,4,38,3,286,...,1.0,2.000000,1.000000,0.0,0.000000,1.0,1.000000,1.000000,0.000000,0.0
6519,2024_02_LA_ARI,2024,2,ARI,REG,37,3,26,4,497,...,0.5,2.000000,2.000000,0.0,0.000000,1.5,1.000000,1.000000,0.500000,0.0
6520,2024_03_DET_ARI,2024,3,ARI,REG,47,8,37,2,284,...,1.0,1.333333,1.666667,0.0,0.333333,1.0,0.666667,0.666667,0.333333,0.0
6521,2024_04_WAS_ARI,2024,4,ARI,REG,43,13,27,1,323,...,1.0,1.250000,1.500000,0.0,0.250000,1.5,0.750000,1.250000,0.250000,0.0
6522,2024_05_ARI_SF,2024,5,ARI,REG,42,0,33,2,364,...,0.8,1.400000,1.400000,0.0,0.400000,1.4,0.600000,1.200000,0.200000,0.2


In [13]:
wdf_def_2024.describe()

,season,week,safety,interception,fumble,fumble_lost,fumble_forced,fumble_not_forced,fumble_out_of_bounds,solo_tackle,...,average_solo_tackle,average_assist_tackle,average_tackle_with_assist,average_sack,average_qb_hit,average_def_touchdown,average_defensive_two_point_attempt,average_defensive_two_point_conv,average_defensive_extra_point_attempt,average_defensive_extra_point_conv
count,570.0,570.000000,570.000000,570.000000,570.000000,570.0000,570.000000,570.000000,570.000000,570.000000,...,570.000000,570.000000,570.000000,570.000000,570.000000,570.000000,570.000000,570.000000,570.0,570.0
mean,2024.0,9.950877,0.029825,0.710526,1.163158,0.5000,0.792982,0.373684,0.100000,35.392982,...,35.101053,15.300547,5.800067,2.504062,5.375857,0.079906,0.006718,0.003369,0.0,0.0
std,0.0,5.606083,0.180280,0.894463,1.070311,0.6818,0.874931,0.634959,0.328227,6.168685,...,2.602829,2.120898,1.787585,0.730125,1.471993,0.113454,0.025772,0.020396,0.0,0.0
min,2024.0,1.000000,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,15.000000,...,22.000000,6.000000,1.000000,0.666667,1.000000,0.000000,0.000000,0.000000,0.0,0.0
25%,2024.0,5.000000,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,32.000000,...,33.719925,14.000000,4.651961,2.000000,4.407143,0.000000,0.000000,0.000000,0.0,0.0
50%,2024.0,10.000000,0.000000,0.000000,1.000000,0.0000,1.000000,0.000000,0.000000,35.000000,...,35.000000,15.062745,5.666667,2.483333,5.250000,0.058824,0.000000,0.000000,0.0,0.0
75%,2024.0,15.000000,0.000000,1.000000,2.000000,1.0000,1.000000,1.000000,0.000000,39.000000,...,36.333333,16.500000,6.633523,2.909091,6.200000,0.111111,0.000000,0.000000,0.0,0.0
max,2024.0,22.000000,2.000000,5.000000,6.000000,4.0000,5.000000,3.000000,2.000000,55.000000,...,46.000000,23.333333,15.000000,6.000000,17.000000,1.000000,0.250000,0.250000,0.0,0.0


In [ ]:
merged_df = pd.merge(wdf_of_2024, wdf_def_2024, on='game_id', how='outer')

In [ ]:
merged_df.describe()

,season_x,week_x,shotgun,no_huddle,qb_dropback,qb_scramble,total_off_yards,pass_attempts,complete_pass,incomplete_pass,...,average_solo_tackle,average_assist_tackle,average_tackle_with_assist,average_sack,average_qb_hit,average_def_touchdown,average_defensive_two_point_attempt,average_defensive_two_point_conv,average_defensive_extra_point_attempt,average_defensive_extra_point_conv
count,1140.0,1140.000000,1140.000000,1140.000000,1140.000000,1140.000000,1140.000000,1140.000000,1140.000000,1140.000000,...,7658.000000,7658.000000,7658.000000,7658.000000,7658.000000,7658.000000,7658.000000,7658.000000,7658.0,7658.0
mean,2024.0,9.950877,46.777193,8.457895,37.085965,2.126316,354.014035,31.885965,21.291228,10.594737,...,39.130104,12.212935,4.424489,2.399433,5.350952,0.128021,0.007047,0.001381,0.0,0.0
std,0.0,5.603621,12.044890,9.387333,8.439922,1.802790,80.697855,7.710931,5.563845,4.199039,...,4.292874,3.451305,2.803911,0.788341,1.356849,0.166167,0.031526,0.013372,0.0,0.0
min,2024.0,1.000000,12.000000,0.000000,14.000000,0.000000,125.000000,10.000000,7.000000,0.000000,...,22.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0
25%,2024.0,5.000000,39.000000,2.000000,31.000000,1.000000,299.000000,27.000000,17.000000,8.000000,...,36.214286,9.833333,2.268182,2.000000,4.500000,0.000000,0.000000,0.000000,0.0,0.0
50%,2024.0,10.000000,47.000000,6.000000,37.000000,2.000000,349.000000,31.000000,21.000000,10.000000,...,39.000000,12.250000,4.142857,2.400000,5.285714,0.083333,0.000000,0.000000,0.0,0.0
75%,2024.0,15.000000,54.000000,11.000000,42.000000,3.000000,411.000000,37.000000,25.000000,13.000000,...,41.833333,14.700000,6.200000,2.857143,6.111111,0.200000,0.000000,0.000000,0.0,0.0
max,2024.0,22.000000,81.000000,64.000000,63.000000,11.000000,645.000000,59.000000,42.000000,27.000000,...,61.000000,28.000000,23.000000,10.000000,17.000000,2.000000,1.000000,0.500000,0.0,0.0
